# 🌾 Tutorial 1 (T1): Data Preprocessing in Python
### Machine Learning for Precision Agriculture (Crop & Fertilizer Datasets)

**Goal:** Learn essential data preprocessing steps before training ML models:
1. **Dataset Inspection & Data Cleaning** (Checking shapes, types, missing values)
2. **Categorical Encoding** (Converting text like `Soil Type` & `Crop Type` into numbers)
3. **Feature Scaling** (`StandardScaler` vs `MinMaxScaler` for nutrient & climate features)
4. **Train-Test Splitting** (80% Training, 20% Testing)

--- 
## ⚙️ Step 0: Import Libraries & Generate Sample Data (If running in Google Colab)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

# Set plot styling
sns.set_theme(style="whitegrid")
plt.rcParams.update({'figure.dpi': 120})

# Ensure data folder exists
os.makedirs("data", exist_ok=True)

# Check if datasets exist, if not create sample datasets for Google Colab instant testing
crop_csv = os.path.join("data", "Crop_recommendation.csv")
fert_csv = os.path.join("data", "Fertilizer_Prediction.csv")

if not os.path.exists(crop_csv):
    print("Creating sample Crop_recommendation.csv for Google Colab...")
    crops = ["rice", "maize", "chickpea", "kidneybeans", "pigeonpeas", "mothbeans", "mungbean", "blackgram", "lentil", "pomegranate"]
    df_c = pd.DataFrame({
        'N': np.random.randint(10, 140, 200),
        'P': np.random.randint(5, 145, 200),
        'K': np.random.randint(15, 205, 200),
        'temperature': np.random.uniform(8.0, 43.0, 200),
        'humidity': np.random.uniform(14.0, 100.0, 200),
        'ph': np.random.uniform(3.5, 9.9, 200),
        'rainfall': np.random.uniform(20.0, 300.0, 200),
        'label': np.random.choice(crops, 200)
    })
    df_c.to_csv(crop_csv, index=False)

if not os.path.exists(fert_csv):
    print("Creating sample Fertilizer_Prediction.csv for Google Colab...")
    ferts = ["Urea", "DAP", "14-35-14", "28-28", "17-17-17", "20-20", "10-26-26"]
    soils = ["Clayey", "Sandy", "Loamy", "Black", "Red"]
    crop_types = ["Maize", "Sugarcane", "Cotton", "Tobacco", "Paddy", "Barley", "Wheat", "Millets"]
    df_f = pd.DataFrame({
        'Temperature': np.random.randint(25, 40, 200),
        'Humidity': np.random.randint(50, 75, 200),
        'Moisture': np.random.randint(25, 70, 200),
        'Soil Type': np.random.choice(soils, 200),
        'Crop Type': np.random.choice(crop_types, 200),
        'Nitrogen': np.random.randint(0, 45, 200),
        'Phosphorous': np.random.randint(0, 45, 200),
        'Potassium': np.random.randint(0, 45, 200),
        'Fertilizer Name': np.random.choice(ferts, 200)
    })
    df_f.to_csv(fert_csv, index=False)

print("[OK] Setup complete! Data files are ready.")

--- 
## 🔍 Step 1: Load and Inspect Datasets
We inspect both datasets to check shape, data types, summary statistics, and missing values.

In [ ]:
# Load Datasets
df_crop = pd.read_csv(crop_csv)
df_fert = pd.read_csv(fert_csv)

print("==================================================")
print("1. CROP RECOMMENDATION DATASET")
print("==================================================")
print("Shape:", df_crop.shape)
print("\nFirst 5 Rows:")
display(df_crop.head())
print("\nMissing Values:\n", df_crop.isnull().sum())

print("\n==================================================")
print("2. FERTILIZER PREDICTION DATASET")
print("==================================================")
print("Shape:", df_fert.shape)
print("\nFirst 5 Rows:")
display(df_fert.head())
print("\nMissing Values:\n", df_fert.isnull().sum())

--- 
## 🏷️ Step 2: Categorical Encoding (Text to Numbers)
Machine Learning algorithms work with numbers, not text strings.  
We use `LabelEncoder` to convert text columns like `Soil Type` and `Crop Type` into numerical integers ($0, 1, 2...$).

In [ ]:
# Encoding Fertilizer dataset categorical variables
df_fert_processed = df_fert.copy()

le_soil = LabelEncoder()
le_crop = LabelEncoder()
le_fert = LabelEncoder()

df_fert_processed['Soil Type_encoded'] = le_soil.fit_transform(df_fert_processed['Soil Type'])
df_fert_processed['Crop Type_encoded'] = le_crop.fit_transform(df_fert_processed['Crop Type'])
df_fert_processed['Fertilizer_encoded'] = le_fert.fit_transform(df_fert_processed['Fertilizer Name'])

print("[OK] Categorical Encoding Complete:")
print("Soil Type Mapping:", dict(zip(le_soil.classes_, le_soil.transform(le_soil.classes_))))
print("Crop Type Mapping:", dict(zip(le_crop.classes_, le_crop.transform(le_crop.classes_))))
display(df_fert_processed[['Soil Type', 'Soil Type_encoded', 'Crop Type', 'Crop Type_encoded']].head())

--- 
## 📏 Step 3: Feature Scaling (StandardScaler vs MinMaxScaler)
In agriculture, Nitrogen ($N$) ranges up to 140, whereas $\text{pH}$ ranges from 3.5 to 9.0.  
- **StandardScaler**: Transforms features to have Mean $\mu = 0$ and Variance $\sigma^2 = 1$ ($z = \frac{x - \mu}{\sigma}$).
- **MinMaxScaler**: Scales features strictly between $0$ and $1$ ($x_{new} = \frac{x - x_{min}}{x_{max} - x_{min}}$).

In [ ]:
numeric_features = ['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall']
X_raw = df_crop[numeric_features]

# Apply StandardScaler
std_scaler = StandardScaler()
X_std = pd.DataFrame(std_scaler.fit_transform(X_raw), columns=numeric_features)

# Apply MinMaxScaler
minmax_scaler = MinMaxScaler()
X_minmax = pd.DataFrame(minmax_scaler.fit_transform(X_raw), columns=numeric_features)

# Plot comparison for Rainfall
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(X_raw['rainfall'], ax=axes[0], color='skyblue', kde=True)
axes[0].set_title("Original Rainfall")

sns.histplot(X_std['rainfall'], ax=axes[1], color='salmon', kde=True)
axes[1].set_title("StandardScaler (Mean=0, Std=1)")

sns.histplot(X_minmax['rainfall'], ax=axes[2], color='lightgreen', kde=True)
axes[2].set_title("MinMaxScaler (Range 0 to 1)")

plt.tight_layout()
plt.show()

--- 
## ✂️ Step 4: Train-Test Split (80% Train, 20% Test)
We split our data so the model trains on 80% of the dataset and is tested on 20% of unseen dataset.

In [ ]:
# Crop dataset train-test split
X = X_std
y = df_crop['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Crop Recommendation Train/Test Split:")
print(f"  Training Samples (80%): {X_train.shape[0]}")
print(f"  Testing Samples  (20%): {X_test.shape[0]}")
print(f"  Features Count:         {X_train.shape[1]}")

--- 
## 💡 What I Learned from Tutorial 1 (Summary for Viva / Notebook)
1. **Data Cleaning**: Checked that there are 0 missing values across all soil & climate columns.
2. **Categorical Encoding**: Used `LabelEncoder` so algorithms can process categorical inputs like `Soil Type` and `Crop Type` as numerical matrices.
3. **Feature Scaling**: Demonstrated that distance-based models require `StandardScaler` to prevent large values (Nitrogen) from dominating smaller values (pH).
4. **Train-Test Split**: Created an 80/20 stratified split to evaluate model accuracy without data leakage.